# Simple Python Decorator and Wrapper

## Learning objective

Understand what a wrapper function does and how the `@decorator` syntax applies that wrapper to another function.

```text
Original function
-> Decorator receives the function
-> Wrapper adds extra behavior
-> Wrapper calls the original function
```

This notebook uses only standard Python. No API keys or external packages are required.


## 1. Start with a normal function

The function below performs one simple task: printing `Hello!`.


In [1]:
def say_hello():
    print("Hello!")


say_hello()


Hello!


## 2. Create a decorator containing a wrapper

The decorator receives the original function as an argument.

The inner `wrapper()` function:

1. Prints a message before the original function.
2. Calls the original function.
3. Prints a message after the original function.
4. Is returned by the decorator.


In [2]:
def add_logging(original_function):

    def wrapper():
        print("Function is starting.")

        original_function()

        print("Function has finished.")

    return wrapper


## 3. Wrap the function manually

The next line passes `say_hello` into `add_logging`.

The returned wrapper is stored in `wrapped_hello`.


In [3]:
wrapped_hello = add_logging(say_hello)

print("Original function:", say_hello)
print("Wrapped function:", wrapped_hello)


Original function: <function say_hello at 0x00000257ADAC5EA0>
Wrapped function: <function add_logging.<locals>.wrapper at 0x00000257ADAC5E10>


## 4. Call the wrapped function

Calling `wrapped_hello()` runs the wrapper. The wrapper then calls the original function internally.


In [4]:
wrapped_hello()


Function is starting.
Hello!
Function has finished.


## 5. Replace the original name manually

A decorator normally replaces the original function name with the returned wrapper.

The statement below means:

```text
Take say_goodbye
-> Pass it to add_logging
-> Store the returned wrapper back in say_goodbye
```


In [5]:
def say_goodbye():
    print("Goodbye!")


say_goodbye = add_logging(say_goodbye)

say_goodbye()


Function is starting.
Goodbye!
Function has finished.


## 6. Use the `@` decorator syntax

The `@add_logging` syntax is a shorter way of performing the same replacement.

This:

```python
@add_logging
def welcome():
    print("Welcome!")
```

is equivalent to:

```python
def welcome():
    print("Welcome!")

welcome = add_logging(welcome)
```


In [6]:
@add_logging
def welcome():
    print("Welcome!")


welcome()


Function is starting.
Welcome!
Function has finished.


## 7. Support functions with arguments

The earlier wrapper accepts no arguments. A reusable wrapper can accept:

- `*args` for positional arguments
- `**kwargs` for named arguments

The wrapper passes those arguments to the original function.


In [7]:
def add_logging_with_arguments(original_function):

    def wrapper(*args, **kwargs):
        print("Function is starting.")
        print("Positional arguments:", args)
        print("Named arguments:", kwargs)

        result = original_function(*args, **kwargs)

        print("Function has finished.")

        return result

    return wrapper


## 8. Decorate a function that returns a value

The wrapper must return the original function's result. Otherwise, the decorated function would return `None`.


In [8]:
@add_logging_with_arguments
def add_numbers(first_number, second_number):
    return first_number + second_number


answer = add_numbers(10, 20)

print("Returned answer:", answer)


Function is starting.
Positional arguments: (10, 20)
Named arguments: {}
Function has finished.
Returned answer: 30


## 9. Call the same function with named arguments

Named arguments are collected inside `kwargs`.


In [9]:
answer = add_numbers(
    first_number=100,
    second_number=50,
)

print("Returned answer:", answer)


Function is starting.
Positional arguments: ()
Named arguments: {'first_number': 100, 'second_number': 50}
Function has finished.
Returned answer: 150


## 10. Inspect the decorated function

After decoration, the function name refers to the wrapper rather than directly to the original function.


In [10]:
print("Displayed function name:", add_numbers.__name__)
print("Function object:", add_numbers)


Displayed function name: wrapper
Function object: <function add_logging_with_arguments.<locals>.wrapper at 0x00000257ADAC6290>


## 11. Preserve the original function's metadata

`functools.wraps` copies useful metadata from the original function to the wrapper.

It also adds the `__wrapped__` attribute used in the ReAct course script to access the original function.


In [11]:
from functools import wraps


def add_logging_preserving_metadata(original_function):

    @wraps(original_function)
    def wrapper(*args, **kwargs):
        print("Function is starting.")

        result = original_function(*args, **kwargs)

        print("Function has finished.")
        return result

    return wrapper


In [12]:
@add_logging_preserving_metadata
def multiply_numbers(first_number, second_number):
    """Multiply two numbers and return the result."""
    return first_number * second_number


print("Function name:", multiply_numbers.__name__)
print("Docstring:", multiply_numbers.__doc__)
print("Original function:", multiply_numbers.__wrapped__)
print("Result:", multiply_numbers(5, 4))


Function name: multiply_numbers
Docstring: Multiply two numbers and return the result.
Original function: <function multiply_numbers at 0x00000257ADAC60E0>
Function is starting.
Function has finished.
Result: 20


## 12. Key takeaway

The most important equivalence is:

```python
@decorator
def my_function():
    pass
```

The code above means:

```python
def my_function():
    pass

my_function = decorator(my_function)
```

The decorator receives the original function and returns a replacement, usually a wrapper.

In the LangChain course:

- `@tool` wraps a Python function and converts it into a LangChain tool.
- `@traceable` wraps a function and adds LangSmith tracing.
- `__wrapped__` can provide access to the original function when the decorator uses `functools.wraps`.
